## 1. Enriquecimiento del Dataset mediante Grafos
Para mejorar la detección de fraude, no solo analizaremos las transacciones de forma aislada, sino también las relaciones entre cuentas. Utilizaremos Neo4j para calcular métricas de centralidad y comunidad que capturen patrones de comportamiento de red.

In [9]:
from neo4j import GraphDatabase
import pandas as pd
import time
import csv

URI = "bolt://neo4j:7687" 
AUTH = ("neo4j", "password")


In [10]:

try:
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        driver.verify_connectivity()
        print("¡Conexión exitosa a tu Neo4j local en Docker! 🚀")
except Exception as e:
    print(f"Error al conectar: {e}")

¡Conexión exitosa a tu Neo4j local en Docker! 🚀


In [11]:
df = pd.read_csv("../data/ml_dataset.csv")

df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [12]:
# Split temporal estricto
df_train = df[df['step'] <= 600].copy()
df_test = df[df['step'] > 600].copy()

# Liberamos el df original para ahorrar RAM
del df

print(f"Train: {df_train.shape[0]} filas | Test: {df_test.shape[0]} filas")

Train: 6259047 filas | Test: 103573 filas


In [13]:
df_train.to_csv("train.csv", index=False)
df_test.to_csv("test.csv", index=False)

print("Archivos exportados correctamente")

Archivos exportados correctamente


In [10]:
def cargar_dataframe_a_neo4j(df, batch_size=10000):
    # Consulta optimizada con UNWIND
    query = """
    UNWIND $rows AS row
    MERGE (orig:Account {id: row.nameOrig})
    MERGE (dest:Account {id: row.nameDest})
    WITH orig, dest, row
    CREATE (orig)-[:TRANSACTION {
        amount: toFloat(row.amount),
        type: row.type,
        step: toInteger(row.step),
        oldbalanceOrg: toFloat(row.oldbalanceOrg),
        newbalanceOrig: toFloat(row.newbalanceOrig),
        oldbalanceDest: toFloat(row.oldbalanceDest),
        newbalanceDest: toFloat(row.newbalanceDest),
        isFraud: toInteger(row.isFraud)
    }]->(dest)
    """

    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            # 1. Creamos el índice (Vital para la velocidad del MERGE)
            print("Asegurando índice en Account(id)...")
            session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (a:Account) REQUIRE a.id IS UNIQUE")
            
            # 2. Preparamos los datos
            # Convertimos el DF a una lista de diccionarios que Neo4j entiende bien
            print("Preparando registros...")
            records = df.to_dict('records')
            total = len(records)
            
            print(f"Iniciando carga de {total} registros en lotes de {batch_size}...")
            start_time = time.time()
            
            # 3. Carga por lotes
            for i in range(0, total, batch_size):
                batch = records[i:i + batch_size]
                session.run(query, rows=batch)
                if i % 100000 == 0 and i > 0:
                    print(f"Progreso: {i}/{total} filas insertadas...")
            
            end_time = time.time()
            print(f"\n--- Carga completada en {round((end_time - start_time)/60, 2)} minutos ---")

# Ejecutar la carga usando tu objeto df_train
cargar_dataframe_a_neo4j(df_train)

Asegurando índice en Account(id)...
Preparando registros...
Iniciando carga de 6259047 registros en lotes de 10000...
Progreso: 100000/6259047 filas insertadas...
Progreso: 200000/6259047 filas insertadas...
Progreso: 300000/6259047 filas insertadas...
Progreso: 400000/6259047 filas insertadas...
Progreso: 500000/6259047 filas insertadas...
Progreso: 600000/6259047 filas insertadas...
Progreso: 700000/6259047 filas insertadas...
Progreso: 800000/6259047 filas insertadas...
Progreso: 900000/6259047 filas insertadas...
Progreso: 1000000/6259047 filas insertadas...
Progreso: 1100000/6259047 filas insertadas...
Progreso: 1200000/6259047 filas insertadas...
Progreso: 1300000/6259047 filas insertadas...
Progreso: 1400000/6259047 filas insertadas...
Progreso: 1500000/6259047 filas insertadas...
Progreso: 1600000/6259047 filas insertadas...
Progreso: 1700000/6259047 filas insertadas...
Progreso: 1800000/6259047 filas insertadas...
Progreso: 1900000/6259047 filas insertadas...
Progreso: 2000000

In [4]:
# 1/4 Configuración y Limpieza
def calcular_grados_enriquecidos():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            print("1. 🧹 Limpiando proyecciones previas...")
            session.run("CALL gds.graph.drop('fraudGraph', false)")
            
            print("2. 🧠 Proyectando el grafo (9M de nodos)...")
            session.run("""
                CALL gds.graph.project(
                    'fraudGraph',
                    'Account',
                    'TRANSACTION'
                )
            """)
            
            print("3. 📤 Calculando Out-Degree (Envíos)...")
            res_out = session.run("""
                CALL gds.degree.write('fraudGraph', {
                    orientation: 'NATURAL',
                    writeProperty: 'out_degree'
                }) YIELD nodePropertiesWritten, computeMillis
            """).single()
            print(f"   -> {res_out['nodePropertiesWritten']:,} cuentas actualizadas (envíos).")
            
            print("4. 📥 Calculando In-Degree (Recepciones)...")
            res_in = session.run("""
                CALL gds.degree.write('fraudGraph', {
                    orientation: 'REVERSE',
                    writeProperty: 'in_degree'
                }) YIELD nodePropertiesWritten, computeMillis
            """).single()
            print(f"   -> {res_in['nodePropertiesWritten']:,} cuentas actualizadas (recepciones).")
            
            print("\n🧹 Liberando memoria RAM...")
            session.run("CALL gds.graph.drop('fraudGraph', false)")

            # --- VERIFICACIÓN DE LOS 2 PRIMEROS REGISTROS ---
            print("\n🔍 Verificación de nuevas columnas (Primeros 2 registros):")
            verificacion = session.run("""
                MATCH (a:Account)
                WHERE a.out_degree IS NOT NULL OR a.in_degree IS NOT NULL
                RETURN a.id AS ID, a.out_degree AS Out, a.in_degree AS In
                LIMIT 2
            """)
            
            for registro in verificacion:
                print(f"ID: {registro['ID']} | Out-Degree: {registro['Out']} | In-Degree: {registro['In']}")

# Ejecutamos la función
calcular_grados_enriquecidos()

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('fraudGraph', false)"


1. 🧹 Limpiando proyecciones previas...
2. 🧠 Proyectando el grafo (9M de nodos)...
3. 📤 Calculando Out-Degree (Envíos)...
   -> 8,924,492 cuentas actualizadas (envíos).
4. 📥 Calculando In-Degree (Recepciones)...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('fraudGraph', false)"


   -> 8,924,492 cuentas actualizadas (recepciones).

🧹 Liberando memoria RAM...

🔍 Verificación de nuevas columnas (Primeros 2 registros):
ID: C1231006815 | Out-Degree: 1.0 | In-Degree: 0.0
ID: C1666544295 | Out-Degree: 1.0 | In-Degree: 0.0


In [5]:
def calcular_pagerank_enriquecido():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            print("1. 🧹 Limpiando proyecciones previas y proyectando...")
            session.run("CALL gds.graph.drop('fraudGraph_pr', false)")
            session.run("""
                CALL gds.graph.project(
                    'fraudGraph_pr',
                    'Account',
                    'TRANSACTION'
                )
            """)
            
            print("2. 🥇 Calculando PageRank (Importancia estructural)...")
            # Aplicamos tus parámetros: 20 iteraciones para convergencia y 0.85 de damping
            res_pr = session.run("""
                CALL gds.pageRank.write('fraudGraph_pr', {
                    maxIterations: 20,
                    dampingFactor: 0.85,
                    writeProperty: 'pagerank'
                }) YIELD nodePropertiesWritten, computeMillis
            """).single()
            
            print(f"   -> {res_pr['nodePropertiesWritten']:,} cuentas actualizadas con PageRank.")
            
            print("\n🧹 Liberando memoria RAM...")
            session.run("CALL gds.graph.drop('fraudGraph_pr', false)")

            # --- VERIFICACIÓN DE LOS 2 PRIMEROS REGISTROS ---
            print("\n🔍 Verificación de PageRank (Primeros 2 registros):")
            verificacion = session.run("""
                MATCH (a:Account)
                WHERE a.pagerank IS NOT NULL
                RETURN a.id AS ID, a.pagerank AS PR
                LIMIT 2
            """)
            
            for registro in verificacion:
                print(f"ID: {registro['ID']} | PageRank: {registro['PR']}")

# Ejecutar el cálculo
calcular_pagerank_enriquecido()

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('fraudGraph_pr', false)"


1. 🧹 Limpiando proyecciones previas y proyectando...
2. 🥇 Calculando PageRank (Importancia estructural)...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('fraudGraph_pr', false)"


   -> 8,924,492 cuentas actualizadas con PageRank.

🧹 Liberando memoria RAM...

🔍 Verificación de PageRank (Primeros 2 registros):
ID: C1231006815 | PageRank: 0.15000000000000002
ID: C1666544295 | PageRank: 0.15000000000000002


In [6]:
def calcular_comunidades_enriquecido():
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            
            print("1/3 🧹 Preparando memoria RAM y proyectando el grafo...")
            session.run("CALL gds.graph.drop('fraudGraph_louvain', false)")
            
            # Proyectamos como UNDIRECTED para capturar la estructura de vecindad completa
            session.run("""
                CALL gds.graph.project(
                    'fraudGraph_louvain',
                    'Account',
                    {
                        TRANSACTION: {
                            orientation: 'UNDIRECTED'
                        }
                    }
                )
            """)
            
            print("2/3 🏘️ Detectando Comunidades (Louvain) y guardando en la BD...")
            res_louvain = session.run("""
                CALL gds.louvain.write('fraudGraph_louvain', {
                    writeProperty: 'community_id'
                }) YIELD nodePropertiesWritten, communityCount, computeMillis
            """).single()
            
            print(f"    -> {res_louvain['nodePropertiesWritten']:,} cuentas actualizadas.")
            print(f"    -> Se detectaron {res_louvain['communityCount']:,} comunidades distintas.")
            
            print("3/3 🧹 Liberando la memoria RAM...")
            session.run("CALL gds.graph.drop('fraudGraph_louvain', false)")

            # --- VERIFICACIÓN DE LOS 2 PRIMEROS REGISTROS ---
            print("\n🔍 Verificación de Comunidades (Primeros 2 registros):")
            verificacion = session.run("""
                MATCH (a:Account)
                WHERE a.community_id IS NOT NULL
                RETURN a.id AS ID, a.community_id AS Comunidad
                LIMIT 2
            """)
            
            for registro in verificacion:
                print(f"ID: {registro['ID']} | ID Comunidad: {registro['Comunidad']}")

# Ejecutar el proceso
calcular_comunidades_enriquecido()

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('fraudGraph_louvain', false)"


1/3 🧹 Preparando memoria RAM y proyectando el grafo...
2/3 🏘️ Detectando Comunidades (Louvain) y guardando en la BD...


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('fraudGraph_louvain', false)"


    -> 8,924,492 cuentas actualizadas.
    -> Se detectaron 2,665,453 comunidades distintas.
3/3 🧹 Liberando la memoria RAM...

🔍 Verificación de Comunidades (Primeros 2 registros):
ID: C1231006815 | ID Comunidad: 53200
ID: C1666544295 | ID Comunidad: 53201


In [19]:
def exportar_dataset_final_v2():
    print("🚀 Iniciando exportación corregida (incluyendo IDs de cuenta)...")
    inicio = time.time()
    
    # Añadimos orig.id y dest.id para poder hacer el merge con el test después
    query_extraccion = """
    MATCH (orig:Account)-[r:TRANSACTION]->(dest:Account)
    RETURN r.step AS step,
           r.type AS type,
           r.amount AS amount,
           orig.id AS nameOrig,        // <--- NUEVA: La llave para el origen
           r.oldbalanceOrg AS oldbalanceOrg,
           r.newbalanceOrig AS newbalanceOrig,
           dest.id AS nameDest,        // <--- NUEVA: La llave para el destino
           r.oldbalanceDest AS oldbalanceDest,
           r.newbalanceDest AS newbalanceDest,
           orig.out_degree AS orig_out_degree,
           orig.in_degree AS orig_in_degree,
           orig.pagerank AS orig_pagerank,
           orig.community_id AS orig_community,
           dest.out_degree AS dest_out_degree,
           dest.in_degree AS dest_in_degree,
           dest.pagerank AS dest_pagerank,
           dest.community_id AS dest_community,
           r.isFraud AS isFraud
    """
    
    try:
        with GraphDatabase.driver(URI, auth=AUTH) as driver:
            with driver.session(fetch_size=10000) as session:
                resultado = session.run(query_extraccion)
                archivo_salida = 'train_enriquecido_final.csv'
                
                with open(archivo_salida, 'w', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(resultado.keys())
                    
                    count = 0
                    for record in resultado:
                        writer.writerow(record.values())
                        count += 1
                        if count % 500000 == 0:
                            print(f"  -> {count:,} registros procesados...")
                            
        fin = time.time()
        print(f"\n✅ ¡Éxito! Dataset generado con IDs: '{archivo_salida}'")
        print(f"📊 Total registros: {count:,}")
        print(f"⏱️ Tiempo total: {(fin - inicio)/60:.2f} minutos.")

    except Exception as e:
        print(f"❌ Error durante la exportación: {e}")

# Ejecutamos la exportación corregida
exportar_dataset_final_v2()

🚀 Iniciando exportación corregida (incluyendo IDs de cuenta)...
  -> 500,000 registros procesados...
  -> 1,000,000 registros procesados...
  -> 1,500,000 registros procesados...
  -> 2,000,000 registros procesados...
  -> 2,500,000 registros procesados...
  -> 3,000,000 registros procesados...
  -> 3,500,000 registros procesados...
  -> 4,000,000 registros procesados...
  -> 4,500,000 registros procesados...
  -> 5,000,000 registros procesados...
  -> 5,500,000 registros procesados...
  -> 6,000,000 registros procesados...

✅ ¡Éxito! Dataset generado con IDs: 'train_enriquecido_final.csv'
📊 Total registros: 6,259,047
⏱️ Tiempo total: 3.74 minutos.


In [20]:
df_train = pd.read_csv("./train_enriquecido_final.csv")
df_test = pd.read_csv("./test.csv")

In [21]:
df_train.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,orig_out_degree,orig_in_degree,orig_pagerank,orig_community,dest_out_degree,dest_in_degree,dest_pagerank,dest_community,isFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,1.0,0.0,0.15,53200,0.0,1.0,0.27750,53200,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,1.0,0.0,0.15,53201,0.0,1.0,0.27750,53201,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1.0,0.0,0.15,6013326,0.0,44.0,5.69625,6013326,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1.0,0.0,0.15,53203,0.0,41.0,5.37750,53203,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,1.0,0.0,0.15,53204,0.0,1.0,0.27750,53204,0


In [22]:

# 1. Crear el Diccionario de Reputación desde el Train
# Combinamos métricas de origen y destino para tener todas las cuentas conocidas
repo_orig = df_train[['nameOrig', 'orig_out_degree', 'orig_in_degree', 'orig_pagerank', 'orig_community']].drop_duplicates(subset=['nameOrig'])
repo_orig.columns = ['AccountID', 'out_degree', 'in_degree', 'pagerank', 'community_id']

repo_dest = df_train[['nameDest', 'dest_out_degree', 'dest_in_degree', 'dest_pagerank', 'dest_community']].drop_duplicates(subset=['nameDest'])
repo_dest.columns = ['AccountID', 'out_degree', 'in_degree', 'pagerank', 'community_id']

df_reputacion = pd.concat([repo_orig, repo_dest]).drop_duplicates(subset=['AccountID'])

# 2. Enriquecer el df_test (mapeo)
df_test_enriquecido = pd.merge(df_test, df_reputacion, left_on='nameOrig', right_on='AccountID', how='left').drop(columns=['AccountID'])
df_test_enriquecido.rename(columns={'out_degree':'orig_out_degree', 'in_degree':'orig_in_degree', 'pagerank':'orig_pagerank', 'community_id':'orig_community'}, inplace=True)

df_test_enriquecido = pd.merge(df_test_enriquecido, df_reputacion, left_on='nameDest', right_on='AccountID', how='left').drop(columns=['AccountID'])
df_test_enriquecido.rename(columns={'out_degree':'dest_out_degree', 'in_degree':'dest_in_degree', 'pagerank':'dest_pagerank', 'community_id':'dest_community'}, inplace=True)

# 3. Fusión total para preprocesamiento unificado
df_train['is_train'] = 1
df_test_enriquecido['is_train'] = 0
df_full = pd.concat([df_train, df_test_enriquecido], axis=0, ignore_index=True)

# 4. Limpieza y Relleno de Nulos (Cuentas nuevas en Test)
fill_values = {
    'orig_in_degree': 0, 'orig_out_degree': 0, 'orig_pagerank': 0.15, 'orig_community': -1,
    'dest_in_degree': 0, 'dest_out_degree': 0, 'dest_pagerank': 0.15, 'dest_community': -1
}
df_full.fillna(value=fill_values, inplace=True)

# Eliminamos duplicados y columnas de "ruido" que analizamos antes
cols_finales_drop = ['nameOrig', 'nameDest', 'orig_in_degree', 'dest_out_degree', 'isFlaggedFraud']
df_full.drop(columns=[c for c in cols_finales_drop if c in df_full.columns], inplace=True)

# 5. Volver a dividir
df_train_final = df_full[df_full['is_train'] == 1].drop(columns=['is_train'])
df_test_final = df_full[df_full['is_train'] == 0].drop(columns=['is_train'])

print(f"✅ ¡Proceso terminado! Columnas finales: {df_train_final.columns.tolist()}")
print(f"Muestras Train: {len(df_train_final)} | Muestras Test: {len(df_test_final)}")

✅ ¡Proceso terminado! Columnas finales: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_out_degree', 'orig_pagerank', 'orig_community', 'dest_in_degree', 'dest_pagerank', 'dest_community', 'isFraud']
Muestras Train: 6259047 | Muestras Test: 103573


In [23]:
df_test_final.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,orig_out_degree,orig_pagerank,orig_community,dest_in_degree,dest_pagerank,dest_community,isFraud
6259047,601,PAYMENT,115.00,50.0,0.00,0.0,0.0,0.0,0.15,-1.0,0.0,0.15,-1.0,0
6259048,601,PAYMENT,23302.85,0.0,0.00,0.0,0.0,0.0,0.15,-1.0,0.0,0.15,-1.0,0
6259049,601,PAYMENT,2180.34,39492.0,37311.66,0.0,0.0,0.0,0.15,-1.0,0.0,0.15,-1.0,0
6259050,601,PAYMENT,1504.97,49660.0,48155.03,0.0,0.0,0.0,0.15,-1.0,0.0,0.15,-1.0,0
6259051,601,PAYMENT,4445.60,1612.0,0.00,0.0,0.0,0.0,0.15,-1.0,0.0,0.15,-1.0,0


In [24]:
df_test_final.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,orig_out_degree,orig_pagerank,orig_community,dest_in_degree,dest_pagerank,dest_community,isFraud
count,103573.000000,1.035730e+05,1.035730e+05,1.035730e+05,1.035730e+05,1.035730e+05,103573.000000,103573.000000,1.035730e+05,103573.000000,103573.000000,1.035730e+05,103573.000000
mean,673.018422,1.814500e+05,6.317498e+05,6.309654e+05,1.151004e+06,1.246841e+06,0.002617,0.150172,8.751523e+03,4.906462,0.775868,2.270282e+06,0.015448
std,29.585853,5.543750e+05,1.962407e+06,1.940836e+06,4.105610e+06,4.172907e+06,0.051085,0.017946,2.115931e+05,7.964956,1.016919,2.806231e+06,0.123327
min,601.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.150000,-1.000000e+00,0.000000,0.150000,-1.000000e+00,0.000000
25%,659.000000,1.245185e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.150000,-1.000000e+00,0.000000,0.150000,-1.000000e+00,0.000000
50%,686.000000,7.272829e+04,2.049800e+04,0.000000e+00,7.887455e+04,1.661894e+05,0.000000,0.150000,-1.000000e+00,1.000000,0.277500,6.167650e+05,0.000000
75%,691.000000,2.057388e+05,1.173820e+05,1.521237e+05,8.354659e+05,9.777057e+05,0.000000,0.150000,-1.000000e+00,7.000000,1.042500,4.358231e+06,0.000000
max,743.000000,1.522288e+07,5.731626e+07,4.731626e+07,3.279981e+08,3.284317e+08,1.000000,3.337500,8.780381e+06,101.000000,12.963750,8.924487e+06,1.000000


In [25]:

# 1. Marcamos los datasets para no perder el control
df_train['is_train'] = 1
df_test_final['is_train'] = 0

# 2. Unificación en un solo bloque
df_full = pd.concat([df_train, df_test_final], axis=0, ignore_index=True)

print(f"📊 Dataset unificado: {df_full.shape[0]:,} filas y {df_full.shape[1]} columnas.")

📊 Dataset unificado: 6,362,620 filas y 19 columnas.


In [27]:
df_full.to_csv("df_final.csv", index=False)
print("💾 Dataset exportado con éxito. ¡Listo para el cuaderno de ML!")

💾 Dataset exportado con éxito. ¡Listo para el cuaderno de ML!
